In [ ]:
!pip install -q gensim xgboost networkx scikit-learn matplotlib seaborn pandas numpy

: 

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os, re, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from collections import Counter

import gensim
from gensim import corpora
from gensim.models import LdaModel

import networkx as nx
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve
)
import xgboost as xgb

warnings.filterwarnings("ignore")

# ── Output directory ──────────────────────────────────────────────
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs("saved_models", exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────
PAL = {
    "blue":"#2C5F8A","orange":"#E07B39","green":"#4CAF50",
    "red":"#E53935","purple":"#8E44AD","dark":"#2C3E50","light":"#F0F4F8",
}
plt.rcParams.update({
    "figure.dpi":150,"savefig.dpi":300,"font.family":"DejaVu Sans",
    "font.size":11,"axes.titlesize":13,"axes.labelsize":11,
    "xtick.labelsize":9,"ytick.labelsize":9,"legend.fontsize":10,
    "axes.spines.top":False,"axes.spines.right":False,
})

def save_fig(fig, filename):
    fig.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300,
                bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"  ✔  Saved → outputs/{filename}")

# ── Stopwords for LDA ─────────────────────────────────────────────
STOPWORDS = {
    "the","a","an","and","or","in","on","of","to","is","it","this","that",
    "was","for","with","as","at","by","from","be","are","has","have","had",
    "were","been","not","no","but","if","so","do","did","does","its","their",
    "there","about","which","all","also","though","involves","reported",
    "claim","amount",
}

def clean_text(text: str) -> list:
    text   = str(text).lower()
    text   = re.sub(r"[^a-z\s]", " ", text)
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 2]
    return tokens

print("✅  Imports and globals ready.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 1 ▸ DATASET LOADING
# ══════════════════════════════════════════════════════════════════

def load_dataset(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)

    print("=" * 55)
    print("  MODULE 1 — DATASET LOADING")
    print("=" * 55)
    print(f"\n  Rows    : {df.shape[0]:,}")
    print(f"  Columns : {df.shape[1]}")
    print(f"\n  Column names:\n  {df.columns.tolist()}")
    print("\n── Dataset Info ──────────────────────────────────────")
    df.info()
    print("\n── First 5 Rows ──────────────────────────────────────")
    display(df.head())

    overview = pd.DataFrame({
        "Column"  : df.columns,
        "Dtype"   : df.dtypes.values.astype(str),
        "Non-Null": df.notna().sum().values,
        "Nulls"   : df.isna().sum().values,
        "Unique"  : df.nunique().values,
    })

    fig, ax = plt.subplots(figsize=(14, len(df.columns) * 0.38 + 1))
    ax.axis("off")
    tbl = ax.table(cellText=overview.values, colLabels=overview.columns,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    tbl.scale(1, 1.45)
    for ci in range(len(overview.columns)):
        tbl[0, ci].set_facecolor(PAL["blue"])
        tbl[0, ci].set_text_props(color="white", fontweight="bold")
    for ri in range(1, len(overview) + 1):
        c = "#EAF2FF" if ri % 2 == 0 else "#FAFAFA"
        for ci in range(len(overview.columns)):
            tbl[ri, ci].set_facecolor(c)

    fig.suptitle("Insurance Claims Dataset — Column Overview",
                 fontsize=14, fontweight="bold", color=PAL["dark"], y=0.98)
    plt.tight_layout()
    save_fig(fig, "dataset_overview.png")
    plt.show()
    return df

# ── RUN ───────────────────────────────────────────────────────────
df_raw = load_dataset("final_dataset(1).csv")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 2 ▸ DATA PREPROCESSING
# Returns df_clean (label-encoded, NOT scaled) + le_target
# Scaling happens later inside train_model, on train split only.
# ══════════════════════════════════════════════════════════════════

def preprocess_data(df: pd.DataFrame):
    print("=" * 55)
    print("  MODULE 2 — DATA PREPROCESSING")
    print("=" * 55)

    df = df.copy()

    # ── Drop columns ──────────────────────────────────────────
    drop_cols = [c for c in df.columns if df[c].isna().all()]
    drop_cols += [c for c in ["policy_number", "insured_zip", "_c39",
                               "policy_bind_date", "incident_date",
                               "incident_location"]
                  if c in df.columns]
    df.drop(columns=list(set(drop_cols)), inplace=True)
    print(f"\n  Dropped  : {list(set(drop_cols))}")
    print(f"  Remaining: {df.shape[1]} columns, {df.shape[0]:,} rows")

    # ── Missing value heatmap ─────────────────────────────────
    mv = df.isnull().sum()
    mv_nonzero = mv[mv > 0].sort_values(ascending=False)
    print("\n── Missing Values (before imputation) ────────────────")
    display(mv_nonzero.rename("Missing Count").to_frame())

    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(df.isnull().astype(int), cbar=False, cmap="Blues",
                ax=ax, xticklabels=True, yticklabels=False)
    ax.set_title("Missing Value Heatmap (before imputation)",
                 fontweight="bold", color=PAL["dark"])
    ax.set_xlabel("Columns"); ax.set_ylabel("Rows")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.tight_layout()
    save_fig(fig, "missing_values_heatmap.png")
    plt.show()

    # ── Impute ────────────────────────────────────────────────
    for col in df.select_dtypes(include="object").columns:
        df[col].fillna(df[col].mode()[0], inplace=True)
    for col in df.select_dtypes(include="number").columns:
        df[col].fillna(df[col].median(), inplace=True)

    # ── Label-encode all categoricals ─────────────────────────
    le_target = None
    for col in df.select_dtypes(include="object").columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        if col == "fraud_reported":
            le_target = le

    # ── Feature distribution plot ─────────────────────────────
    num_cols = [c for c in df.select_dtypes(include="number").columns
                if c != "fraud_reported"][:12]
    fig, axes = plt.subplots(3, 4, figsize=(16, 10))
    for ax, col in zip(axes.flat, num_cols):
        ax.hist(df[col], bins=25, color=PAL["blue"], edgecolor="white", alpha=0.8)
        ax.set_title(col, fontsize=9); ax.set_xlabel("")
    for ax in axes.flat[len(num_cols):]:
        ax.set_visible(False)
    fig.suptitle("Feature Distributions (label-encoded, pre-split)",
                 fontweight="bold", color=PAL["dark"])
    plt.tight_layout()
    save_fig(fig, "feature_distribution.png")
    plt.show()

    print(f"\n  ✔  df_clean shape : {df.shape}  (NOT yet scaled)")
    return df, le_target

# ── RUN ───────────────────────────────────────────────────────────
df_clean, le_target = preprocess_data(df_raw)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 3 ▸ LDA TOPIC MODELLING  (definition only — called inside train_model)
# ══════════════════════════════════════════════════════════════════

def perform_lda(df_raw, df_clean, train_idx, test_idx,
                n_topics=5, passes=15):
    """
    Fit dictionary + LDA model on TRAIN rows only.
    Transform both train and test using those fitted objects.
    """
    print("=" * 55)
    print("  MODULE 3 — LDA TOPIC MODELLING  (leak-free)")
    print("=" * 55)

    all_texts   = df_raw["claim_description"].fillna("").apply(clean_text).tolist()
    train_texts = [all_texts[i] for i in train_idx]
    test_texts  = [all_texts[i] for i in test_idx]

    # Fit dictionary on TRAIN only
    dictionary = corpora.Dictionary(train_texts)
    dictionary.filter_extremes(no_below=2, no_above=0.95)
    print(f"\n  Topics requested : {n_topics}")
    print(f"  Dictionary size  : {len(dictionary):,} tokens  (train only)")

    train_corpus = [dictionary.doc2bow(t) for t in train_texts]
    test_corpus  = [dictionary.doc2bow(t) for t in test_texts]

    # Train LDA on TRAIN corpus only
    lda_model = LdaModel(
        corpus=train_corpus, id2word=dictionary,
        num_topics=n_topics, passes=passes,
        random_state=42, alpha="auto", eta="auto",
    )

    def corpus_to_topic_df(corpus, label):
        rows = []
        for doc_bow in corpus:
            dist = dict(lda_model.get_document_topics(doc_bow, minimum_probability=0.0))
            rows.append([dist.get(t, 0.0) for t in range(n_topics)])
        df_out = pd.DataFrame(rows, columns=[f"topic_{i+1}" for i in range(n_topics)])
        print(f"  {label} topic features: {df_out.shape}")
        return df_out

    lda_train = corpus_to_topic_df(train_corpus, "Train")
    lda_test  = corpus_to_topic_df(test_corpus,  "Test ")

    # Visualise top words per topic
    fig, axes = plt.subplots(1, n_topics, figsize=(4 * n_topics, 4))
    for t, ax in enumerate(axes):
        words = lda_model.show_topic(t, topn=8)
        terms, weights = zip(*words)
        ax.barh(terms[::-1], weights[::-1], color=PAL["blue"])
        ax.set_title(f"Topic {t+1}", fontweight="bold")
        ax.set_xlabel("Weight")
    fig.suptitle("LDA Top Words per Topic  (fit on train set only)",
                 fontweight="bold", color=PAL["dark"])
    plt.tight_layout()
    save_fig(fig, "lda_topics.png")
    plt.show()

    # Topic distribution plot
    topic_means = lda_train.mean()
    fig2, ax2 = plt.subplots(figsize=(8, 4))
    ax2.bar(topic_means.index, topic_means.values, color=PAL["blue"], edgecolor="white")
    ax2.set_title("Average Topic Distribution (train set)",
                  fontweight="bold", color=PAL["dark"])
    ax2.set_xlabel("Topic"); ax2.set_ylabel("Mean Probability")
    plt.tight_layout()
    save_fig(fig2, "topic_distribution.png")
    plt.show()

    return lda_train, lda_test, lda_model, dictionary

print("✅  perform_lda defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 4 ▸ GRAPH CONSTRUCTION  (definition only — called inside train_model)
# ══════════════════════════════════════════════════════════════════

def build_graph(df_raw_train: pd.DataFrame) -> nx.Graph:
    """Build co-occurrence graph from TRAINING rows only."""
    print("=" * 55)
    print("  MODULE 4 — GRAPH CONSTRUCTION  (train rows only)")
    print("=" * 55)

    G = nx.Graph()
    graph_cols = [c for c in ["incident_city", "incident_type",
                               "insured_relationship", "policy_state"]
                  if c in df_raw_train.columns]
    print(f"\n  Source columns : {graph_cols}")
    print(f"  Rows used      : {len(df_raw_train):,}  (train only)")

    for _, row in df_raw_train[graph_cols].iterrows():
        values = [str(row[c]) for c in graph_cols]
        for i in range(len(values)):
            for j in range(i + 1, len(values)):
                u, v = values[i], values[j]
                if G.has_edge(u, v):
                    G[u][v]["weight"] += 1
                else:
                    G.add_edge(u, v, weight=1)

    print(f"\n   Nodes : {G.number_of_nodes():,}")
    print(f"   Edges : {G.number_of_edges():,}")

    top10 = sorted(dict(G.degree()).items(), key=lambda x: x[1], reverse=True)[:10]
    display(pd.DataFrame(top10, columns=["Node", "Degree"]))
    return G

print("✅  build_graph defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 5 ▸ GRAPH VISUALISATION  (definition only — called inside train_model)
# ══════════════════════════════════════════════════════════════════

def visualize_graph(G: nx.Graph) -> None:
    print("=" * 55)
    print("  MODULE 5 — GRAPH VISUALISATION")
    print("=" * 55)

    degree_dict = dict(G.degree())
    top_nodes   = sorted(degree_dict, key=degree_dict.get, reverse=True)[:25]
    sub_G       = G.subgraph(top_nodes).copy()

    print(f"\nFull Graph : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    print(f"Displayed  : {sub_G.number_of_nodes()} nodes, {sub_G.number_of_edges()} edges")

    pos          = nx.spring_layout(sub_G, seed=42, k=5, iterations=250)
    degrees      = [sub_G.degree(n) for n in sub_G.nodes()]
    node_sizes   = [max(1800, d * 400) for d in degrees]
    edge_weights = [sub_G[u][v].get("weight", 1) for u, v in sub_G.edges()]
    max_w        = max(edge_weights) if edge_weights else 1
    edge_widths  = [1 + 5 * (w / max_w) for w in edge_weights]

    fig, ax = plt.subplots(figsize=(26, 20))
    ax.set_facecolor("#F8FBFF"); fig.patch.set_facecolor("#F8FBFF")

    nx.draw_networkx_edges(sub_G, pos, width=edge_widths,
                           alpha=0.35, edge_color="#999999", ax=ax)
    nx.draw_networkx_nodes(sub_G, pos, node_size=node_sizes,
                           node_color="#2C5F8A", edgecolors="black",
                           linewidths=1.8, alpha=0.95, ax=ax)

    short_labels = {}
    for node in sub_G.nodes():
        label = str(node)
        for prefix in ["incident_city_","incident_type_",
                        "insured_relationship_","policy_state_"]:
            label = label.replace(prefix, "")
        short_labels[node] = label[:15] + ("..." if len(label) > 15 else "")

    nx.draw_networkx_labels(sub_G, pos, labels=short_labels,
                            font_size=12, font_weight="bold",
                            font_color="white", ax=ax)

    ax.set_title("Insurance Claims Relationship Network\nTop 25 Most Connected Entities",
                 fontsize=20, fontweight="bold", color="#2C3E50", pad=20)
    ax.axis("off")
    plt.tight_layout()
    save_fig(fig, "insurance_network_graph.png")
    plt.show()

    print("\nTop Connected Entities"); print("-" * 40)
    for node, deg in sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:10]:
        label = str(node)
        for prefix in ["incident_city_","incident_type_",
                        "insured_relationship_","policy_state_"]:
            label = label.replace(prefix, "")
        print(f"{label:<25} Degree: {deg}")

print("✅  visualize_graph defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 6 ▸ GRAPH FEATURE EXTRACTION  (definition only — called inside train_model)
# ══════════════════════════════════════════════════════════════════

def extract_graph_features(G, df_raw_train, df_raw_test):
    """
    Compute centrality from TRAIN graph only.
    Test rows look up the same pre-computed stats — no test info used.
    Unknown cities fall back to training-set mean.
    """
    print("=" * 55)
    print("  MODULE 6 — GRAPH FEATURE EXTRACTION  (leak-free)")
    print("=" * 55)

    degree_cent = nx.degree_centrality(G)
    clustering  = nx.clustering(G)
    pagerank    = nx.pagerank(G, alpha=0.85)

    mean_dc = float(np.mean(list(degree_cent.values())))
    mean_cc = float(np.mean(list(clustering.values())))
    mean_pr = float(np.mean(list(pagerank.values())))

    def map_feats(df_split):
        city = df_split["incident_city"].astype(str)
        return pd.DataFrame({
            "graph_degree_centrality": city.map(lambda c: degree_cent.get(c, mean_dc)),
            "graph_clustering_coeff" : city.map(lambda c: clustering.get(c, mean_cc)),
            "graph_pagerank"         : city.map(lambda c: pagerank.get(c, mean_pr)),
        }).reset_index(drop=True)

    gf_train = map_feats(df_raw_train)
    gf_test  = map_feats(df_raw_test)

    print(f"  Train graph features : {gf_train.shape}")
    print(f"  Test  graph features : {gf_test.shape}")

    # Centrality chart
    top_nodes = sorted(degree_cent.items(), key=lambda x: x[1], reverse=True)[:15]
    nodes_s, dc_vals = zip(*top_nodes)
    cc_vals = [clustering.get(n, 0) for n in nodes_s]
    pr_vals = [pagerank.get(n, 0)   for n in nodes_s]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, vals, title, colour in zip(
            axes,
            [dc_vals, cc_vals, pr_vals],
            ["Degree Centrality","Clustering Coeff","PageRank"],
            [PAL["blue"], PAL["orange"], PAL["green"]]):
        ax.barh([str(n)[:15] for n in nodes_s[::-1]], list(vals)[::-1],
                color=colour, edgecolor="white")
        ax.set_title(title, fontweight="bold", color=PAL["dark"])
    fig.suptitle("Top 15 Nodes — Graph Centrality Metrics (train only)",
                 fontweight="bold", color=PAL["dark"])
    plt.tight_layout()
    save_fig(fig, "centrality_chart.png")
    plt.show()

    # Graph features sample
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    ax2.axis("off")
    tbl = ax2.table(cellText=gf_train.head(10).round(4).values,
                    colLabels=gf_train.columns,
                    cellLoc="center", loc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.4)
    ax2.set_title("Graph Features — First 10 Train Rows",
                  fontweight="bold", color=PAL["dark"], pad=12)
    plt.tight_layout()
    save_fig(fig2, "graph_features.png")
    plt.show()

    return gf_train, gf_test

print("✅  extract_graph_features defined.")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 8 ▸ MODEL TRAINING — leak-free orchestrator
# Order: split → scale (train) → LDA (train) → graph (train) → fuse → XGBoost
# ══════════════════════════════════════════════════════════════════

def train_model(df_clean, df_raw, test_size=0.3, random_state=42):
    print("=" * 55)
    print("  MODULE 8 — MODEL TRAINING  (leak-free pipeline)")
    print("=" * 55)

    # STEP 1 — Split indices FIRST ────────────────────────────
    target_col = "fraud_reported"
    y_all      = df_clean[target_col].reset_index(drop=True)
    all_idx    = np.arange(len(df_clean))

    train_idx, test_idx = train_test_split(
        all_idx, test_size=test_size,
        random_state=random_state, stratify=y_all)

    print(f"\n  Training samples : {len(train_idx):,}")
    print(f"  Test samples     : {len(test_idx):,}")
    print(f"  Fraud rate (all) : {y_all.mean():.2%}")

    # STEP 2 — Scale: fit on TRAIN only ───────────────────────
    drop_for_X  = [c for c in [target_col, "claim_description"]
                   if c in df_clean.columns]
    structured  = df_clean.drop(columns=drop_for_X).reset_index(drop=True)

    scaler = StandardScaler()
    structured_train = pd.DataFrame(
        scaler.fit_transform(structured.iloc[train_idx]),
        columns=structured.columns)
    structured_test  = pd.DataFrame(
        scaler.transform(structured.iloc[test_idx]),
        columns=structured.columns)

    # STEP 3 — LDA: fit on TRAIN only ─────────────────────────
    lda_train, lda_test, lda_model, dictionary = perform_lda(
        df_raw, df_clean, train_idx, test_idx, n_topics=5, passes=15)

    # STEP 4 — Graph: build on TRAIN rows only ────────────────
    df_raw_train = df_raw.iloc[train_idx].reset_index(drop=True)
    df_raw_test  = df_raw.iloc[test_idx].reset_index(drop=True)
    G = build_graph(df_raw_train)
    visualize_graph(G)
    gf_train, gf_test = extract_graph_features(G, df_raw_train, df_raw_test)

    # STEP 5 — Fuse features ──────────────────────────────────
    X_train = pd.concat([structured_train, lda_train, gf_train], axis=1)
    X_test  = pd.concat([structured_test,  lda_test,  gf_test],  axis=1)
    y_train = y_all.iloc[train_idx].reset_index(drop=True)
    y_test  = y_all.iloc[test_idx].reset_index(drop=True)

    print(f"\n  X_train : {X_train.shape}   X_test : {X_test.shape}")

    # Correlation heatmap on TRAIN features only
    corr_df  = X_train.copy()
    corr_df["fraud_reported"] = y_train.values
    top_cols = (corr_df.corr()["fraud_reported"].abs()
                .sort_values(ascending=False).head(21).index)
    corr_mat = corr_df[top_cols].corr()
    fig, ax  = plt.subplots(figsize=(14, 11))
    mask = np.triu(np.ones_like(corr_mat, dtype=bool))
    sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f",
                cmap="coolwarm", center=0, linewidths=0.4,
                ax=ax, annot_kws={"size": 6})
    ax.set_title("Feature Correlation Heatmap (Top 20 + target, train only)",
                 fontweight="bold", color=PAL["dark"])
    plt.xticks(rotation=45, ha="right", fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout()
    save_fig(fig, "feature_correlation_heatmap.png")
    plt.show()

    # STEP 6 — Train XGBoost ──────────────────────────────────
    scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
    model = xgb.XGBClassifier(
        n_estimators=30, max_depth=2, learning_rate=0.2,
        subsample=0.5, colsample_bytree=0.5, min_child_weight=15,
        gamma=1.0, reg_alpha=2.0, reg_lambda=5.0,
        scale_pos_weight=scale_pos, eval_metric="logloss",
        random_state=random_state, verbosity=0)

    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)], verbose=False)

    train_acc = (model.predict(X_train) == y_train).mean()
    test_acc  = (model.predict(X_test)  == y_test).mean()
    print(f"\n  Training Accuracy : {train_acc:.4f}")
    print(f"  Test     Accuracy : {test_acc:.4f}")

    # Feature importance plot
    fi_df = (pd.DataFrame({"Feature": X_train.columns,
                            "Importance": model.feature_importances_})
             .sort_values("Importance", ascending=False).head(20))
    colour_bands = ([PAL["blue"]] * 5 + [PAL["orange"]] * 5 + [PAL["green"]] * 10)
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1],
            color=colour_bands[:len(fi_df)], edgecolor="white")
    for bar in ax.patches:
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{bar.get_width():.4f}", va="center", fontsize=7.5)
    ax.set_xlabel("Feature Importance (gain)")
    ax.set_title("Top 20 Feature Importances — XGBoost",
                 fontweight="bold", color=PAL["dark"])
    plt.tight_layout()
    save_fig(fig, "feature_importance.png")
    plt.show()

    return (model, X_train, X_test, y_train, y_test,
            scaler, lda_model, dictionary, G, train_idx, test_idx)

# ── RUN ───────────────────────────────────────────────────────────
(model, X_train, X_test, y_train, y_test,
 scaler, lda_model, dictionary, G,
 train_idx, test_idx) = train_model(df_clean, df_raw, test_size=0.3, random_state=7)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 9 ▸ MODEL EVALUATION
# ══════════════════════════════════════════════════════════════════

def evaluate_model(model, X_test, y_test) -> dict:
    print("=" * 55)
    print("  MODULE 9 — MODEL EVALUATION")
    print("=" * 55)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall"   : recall_score(y_test, y_pred, zero_division=0),
        "F1 Score" : f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC"  : roc_auc_score(y_test, y_prob),
    }

    print("\n── Evaluation Metrics ────────────────────────────────")
    met_df = pd.DataFrame({
        "Metric" : list(metrics.keys()),
        "Score"  : [f"{v:.4f}" for v in metrics.values()],
        "Score %": [f"{v:.2%}"  for v in metrics.values()],
    })
    display(met_df.style.hide(axis="index"))

    cm  = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Side-by-side confusion matrix + ROC
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Genuine","Fraud"],
                yticklabels=["Genuine","Fraud"],
                linewidths=1, ax=axes[0],
                annot_kws={"size": 16, "fontweight": "bold"})
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
    axes[0].set_title("Confusion Matrix", fontweight="bold", color=PAL["dark"])

    axes[1].plot(fpr, tpr, color=PAL["blue"], lw=2,
                 label=f"AUC = {metrics['ROC-AUC']:.4f}")
    axes[1].plot([0,1],[0,1],"--", color="#AAAAAA", lw=1.5, label="Random")
    axes[1].fill_between(fpr, tpr, alpha=0.08, color=PAL["blue"])
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title("ROC Curve", fontweight="bold", color=PAL["dark"])
    axes[1].legend(loc="lower right")
    axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1.02])

    fig.suptitle("Model Evaluation — Insurance Fraud Detection",
                 fontsize=14, fontweight="bold", color=PAL["dark"])
    plt.tight_layout()
    save_fig(fig, "confusion_matrix_roc.png")
    plt.show()

    # Individual confusion matrix
    f2, a2 = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Genuine","Fraud"],
                yticklabels=["Genuine","Fraud"],
                linewidths=1, ax=a2,
                annot_kws={"size":16,"fontweight":"bold"})
    a2.set_xlabel("Predicted"); a2.set_ylabel("Actual")
    plt.tight_layout(); save_fig(f2, "confusion_matrix.png"); plt.close(f2)

    # Individual ROC
    f3, a3 = plt.subplots(figsize=(7, 6))
    a3.plot(fpr, tpr, color=PAL["blue"], lw=2,
            label=f"AUC = {metrics['ROC-AUC']:.4f}")
    a3.plot([0,1],[0,1],"--",color="#AAAAAA",lw=1.5)
    a3.fill_between(fpr, tpr, alpha=0.08, color=PAL["blue"])
    a3.set_xlabel("FPR"); a3.set_ylabel("TPR")
    a3.set_title("ROC Curve", fontweight="bold"); a3.legend()
    plt.tight_layout(); save_fig(f3, "roc_curve.png"); plt.close(f3)

    return metrics

# ── RUN ───────────────────────────────────────────────────────────
metrics = evaluate_model(model, X_test, y_test)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 10 ▸ PREDICTION DEMO
# Predicts on X_test rows (unseen during training).
# Change claim_index (0 to len(X_test)-1) to test any test row.
# ══════════════════════════════════════════════════════════════════

def predict_claim(model, X_test, df_raw_test, claim_index=0) -> dict:
    print("=" * 55)
    print(f"  MODULE 10 — PREDICTION DEMO  (test row index: {claim_index})")
    print("=" * 55)

    sample = X_test.iloc[[claim_index]]
    prob   = model.predict_proba(sample)[0, 1]
    pred   = int(prob >= 0.5)

    label  = "🔴  FRAUDULENT CLAIM" if pred == 1 else "🟢  GENUINE CLAIM"
    colour = PAL["red"] if pred == 1 else PAL["green"]
    desc   = (df_raw_test["claim_description"].iloc[claim_index]
              if "claim_description" in df_raw_test.columns else "N/A")

    print(f"\n  Description : {desc}")
    print(f"\n  {'─'*45}")
    print(f"  Result      : {label}")
    print(f"  P(Fraud)    : {prob:.4f}  ({prob:.2%})")
    print(f"  P(Genuine)  : {1-prob:.4f}  ({1-prob:.2%})")
    print(f"  {'─'*45}")

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.set_facecolor(colour); fig.patch.set_facecolor(colour); ax.axis("off")
    ax.text(0.5, 0.72, label, transform=ax.transAxes, ha="center",
            fontsize=24, fontweight="bold", color="white")
    ax.text(0.5, 0.47,
            f"P(Fraud) = {prob:.4f}    |    P(Genuine) = {1-prob:.4f}",
            transform=ax.transAxes, ha="center",
            fontsize=13, color="white", alpha=0.9)
    snippet = desc[:100] + ("..." if len(desc) > 100 else "")
    ax.text(0.5, 0.20, f'Test row #{claim_index}: "{snippet}"',
            transform=ax.transAxes, ha="center",
            fontsize=9, color="white", alpha=0.8)
    plt.tight_layout()
    plt.show()
    return {"label": label, "p_fraud": float(prob), "p_genuine": float(1 - prob)}

# ── RUN — uses test rows only ─────────────────────────────────────
df_raw_test = df_raw.iloc[test_idx].reset_index(drop=True)
result0 = predict_claim(model, X_test, df_raw_test, claim_index=0)
result1 = predict_claim(model, X_test, df_raw_test, claim_index=7)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# MODULE 11 ▸ RESULTS DASHBOARD
# ══════════════════════════════════════════════════════════════════

def generate_results_dashboard(metrics: dict) -> None:
    print("=" * 55)
    print("  MODULE 11 — RESULTS DASHBOARD")
    print("=" * 55)

    card_colours = {
        "Accuracy" : PAL["blue"],
        "Precision": PAL["orange"],
        "Recall"   : PAL["green"],
        "F1 Score" : PAL["purple"],
        "ROC-AUC"  : PAL["red"],
    }

    fig = plt.figure(figsize=(18, 9))
    fig.patch.set_facecolor(PAL["light"])
    gs  = gridspec.GridSpec(2, 5, figure=fig,
                            hspace=0.55, wspace=0.30,
                            top=0.80, bottom=0.15,
                            left=0.04, right=0.98)

    for col, (name, val) in enumerate(metrics.items()):
        ax = fig.add_subplot(gs[0, col])
        ax.set_facecolor(card_colours[name])
        for sp in ax.spines.values(): sp.set_visible(False)
        ax.set_xticks([]); ax.set_yticks([])
        ax.text(0.5, 0.62, f"{val:.2%}", transform=ax.transAxes, ha="center",
                fontsize=24, fontweight="bold", color="white")
        ax.text(0.5, 0.22, name, transform=ax.transAxes, ha="center",
                fontsize=12, color="white", alpha=0.92)

    ax_bar  = fig.add_subplot(gs[1, :])
    ax_bar.set_facecolor("white")
    names   = list(metrics.keys())
    values  = list(metrics.values())
    colours = list(card_colours.values())
    bars = ax_bar.barh(names[::-1], values[::-1],
                       color=colours[::-1], height=0.50, edgecolor="white")
    for bar, val in zip(bars, values[::-1]):
        ax_bar.text(bar.get_width() + 0.005,
                    bar.get_y() + bar.get_height() / 2,
                    f"{val:.4f}", va="center",
                    fontsize=11, fontweight="bold", color=PAL["dark"])
    ax_bar.set_xlim(0, 1.14)
    ax_bar.set_xlabel("Score", fontsize=12)
    ax_bar.set_title("Model Performance Summary",
                     fontweight="bold", fontsize=13, color=PAL["dark"])
    ax_bar.axvline(x=0.9, color="#AAAAAA", linestyle="--",
                   lw=1.5, label="90 % threshold")
    ax_bar.legend(loc="lower right", fontsize=10)

    fig.suptitle(
        "Insurance Fraud Detection — Final Results Dashboard\n"
        "XGBoost  +  LDA Topic Modelling  +  Graph Features",
        fontsize=15, fontweight="bold", color=PAL["dark"], y=0.97,
    )
    save_fig(fig, "final_dashboard.png")
    plt.show()

    print("\n── Final Metrics Summary ─────────────────────────────")
    summary = pd.DataFrame({
        "Metric"  : list(metrics.keys()),
        "Score"   : [f"{v:.4f}" for v in metrics.values()],
        "Score %": [f"{v:.2%}"  for v in metrics.values()],
    })
    display(summary.style.hide(axis="index"))

    print("\n── Output Files Saved ────────────────────────────────")
    for f in ["dataset_overview.png","missing_values_heatmap.png",
              "feature_distribution.png","lda_topics.png",
              "topic_distribution.png","insurance_network_graph.png",
              "centrality_chart.png","graph_features.png",
              "feature_correlation_heatmap.png","feature_importance.png",
              "confusion_matrix_roc.png","confusion_matrix.png",
              "roc_curve.png","final_dashboard.png"]:
        print(f"    ✔  outputs/{f}")

# ── RUN ───────────────────────────────────────────────────────────
generate_results_dashboard(metrics)

In [ ]:
# ══════════════════════════════════════════════════════════════════
# SAVE ALL MODEL ARTIFACTS
# ══════════════════════════════════════════════════════════════════

os.makedirs("saved_models", exist_ok=True)

# XGBoost model
joblib.dump(model, "saved_models/xgboost_fraud_model.pkl")
print("✔  xgboost_fraud_model.pkl")

# StandardScaler
joblib.dump(scaler, "saved_models/scaler.pkl")
print("✔  scaler.pkl")

# Target label encoder classes
joblib.dump(le_target.classes_, "saved_models/target_classes.pkl")
print("✔  target_classes.pkl")

# LDA model (gensim save)
lda_model.save("saved_models/lda_model.gensim")
print("✔  lda_model.gensim")

# Gensim dictionary
dictionary.save("saved_models/lda_dictionary.dict")
print("✔  lda_dictionary.dict")

# NetworkX graph
with open("saved_models/insurance_graph.pkl", "wb") as f:
    pickle.dump(G, f)
print("✔  insurance_graph.pkl")

# Feature column names (from X_train)
joblib.dump(list(X_train.columns), "saved_models/feature_columns.pkl")
print("✔  feature_columns.pkl")

# Train/test split indices (for reproducibility)
joblib.dump({"train_idx": train_idx, "test_idx": test_idx},
            "saved_models/split_indices.pkl")
print("✔  split_indices.pkl")

print("\n✅  ALL FILES SAVED to saved_models/")

In [ ]:
# ══════════════════════════════════════════════════════════════════
# DOWNLOAD — zip everything and download in one click
# ══════════════════════════════════════════════════════════════════

import shutil
from google.colab import files

shutil.make_archive("saved_models_complete", "zip", "saved_models")
files.download("saved_models_complete.zip")